In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(r"D:\Aircraft-Maintenance-Analytics\data\raw\2009.csv",low_memory=False)

print(df.shape)

(6429338, 28)


In [3]:
#Remove Empty Column
df.drop(
    columns=["Unnamed: 27"],
    inplace=True
)

print(df.shape)

(6429338, 27)


In [4]:
#Check Duplicates
duplicates = df.duplicated().sum()

print("Duplicate Records:", duplicates)


Duplicate Records: 510852


In [5]:
#Remove Duplicates
df = df.drop_duplicates()

print(df.shape)


(5918486, 27)


In [6]:
#Convert Date Column
df["FL_DATE"] = pd.to_datetime(
    df["FL_DATE"]
)


In [7]:
#Create Calendar Features
df["YEAR"] = df["FL_DATE"].dt.year

df["MONTH"] = df["FL_DATE"].dt.month

df["DAY"] = df["FL_DATE"].dt.day

df["DAY_NAME"] = df["FL_DATE"].dt.day_name()


In [8]:
#Missing Values Analysis
missing = (
    df.isnull()
      .sum()
      .sort_values(ascending=False)
)

missing


CANCELLATION_CODE      5834382
SECURITY_DELAY         4814747
NAS_DELAY              4814747
WEATHER_DELAY          4814747
LATE_AIRCRAFT_DELAY    4814747
CARRIER_DELAY          4814747
ACTUAL_ELAPSED_TIME      98487
AIR_TIME                 98486
ARR_DELAY                98486
WHEELS_ON                86292
ARR_TIME                 86292
TAXI_IN                  86291
TAXI_OUT                 82912
WHEELS_OFF               82912
DEP_TIME                 80102
DEP_DELAY                80102
CRS_ARR_TIME                 0
OP_CARRIER                   0
ORIGIN                       0
OP_CARRIER_FL_NUM            0
CRS_DEP_TIME                 0
DEST                         0
FL_DATE                      0
DISTANCE                     0
CANCELLED                    0
DIVERTED                     0
CRS_ELAPSED_TIME             0
YEAR                         0
MONTH                        0
DAY                          0
DAY_NAME                     0
dtype: int64

In [9]:
#Fill With Zero
delay_reason_cols = [
    "CARRIER_DELAY",
    "WEATHER_DELAY",
    "NAS_DELAY",
    "SECURITY_DELAY",
    "LATE_AIRCRAFT_DELAY"
]

df[delay_reason_cols] = (
    df[delay_reason_cols]
    .fillna(0)
)

In [10]:
df[
    df["ARR_DELAY"].isnull()
][["CANCELLED", "DIVERTED"]].value_counts()

CANCELLED  DIVERTED
1.0        0.0         84104
0.0        1.0         14382
Name: count, dtype: int64

In [11]:
df[
    df["DEP_DELAY"].isnull()
][["CANCELLED", "DIVERTED"]].value_counts()

CANCELLED  DIVERTED
1.0        0.0         80102
Name: count, dtype: int64

In [12]:
#Create Flight Status
def flight_status(row):

    if row["CANCELLED"] == 1:
        return "Cancelled"

    elif row["DIVERTED"] == 1:
        return "Diverted"

    elif row["ARR_DELAY"] <= 15:
        return "On Time"

    else:
        return "Delayed"


In [13]:
df["FLIGHT_STATUS"] = df.apply(
    flight_status,
    axis=1
)

In [14]:
df["FLIGHT_STATUS"].value_counts()

FLIGHT_STATUS
On Time      4760591
Delayed      1059409
Cancelled      84104
Diverted       14382
Name: count, dtype: int64

In [15]:
#Create On-Time Flag
df["ON_TIME_FLAG"] = np.where(
    df["FLIGHT_STATUS"] == "On Time",
    1,
    0
)


In [16]:
df["ON_TIME_FLAG"].value_counts()

ON_TIME_FLAG
1    4760591
0    1157895
Name: count, dtype: int64

In [18]:
#Create Delay Category
def delay_category(delay):

    if pd.isna(delay):
        return "Not Available"

    elif delay <= 15:
        return "On Time"

    elif delay <= 60:
        return "Minor Delay"

    elif delay <= 180:
        return "Moderate Delay"

    else:
        return "Severe Delay"


In [19]:
df["DELAY_CATEGORY"] = (
    df["ARR_DELAY"]
      .apply(delay_category)
)

In [20]:
df["DELAY_CATEGORY"].value_counts()

DELAY_CATEGORY
On Time           4760591
Minor Delay        749579
Moderate Delay     273521
Not Available       98486
Severe Delay        36309
Name: count, dtype: int64

In [22]:
#Memory Optimization
float_cols = df.select_dtypes(
    include=["float64"]
).columns

for col in float_cols:
    df[col] = df[col].astype("float32")


In [23]:
int_cols = df.select_dtypes(
    include=["int64"]
).columns

for col in int_cols:
    df[col] = df[col].astype("int32")

In [24]:
#Final Validation
print(df.shape)


(5918486, 34)


In [25]:
print(
    "Duplicates:",
    df.duplicated().sum()
)

Duplicates: 0


In [26]:
df.isnull().sum().sort_values(
    ascending=False
).head(15)

CANCELLATION_CODE      5834382
ACTUAL_ELAPSED_TIME      98487
AIR_TIME                 98486
ARR_DELAY                98486
ARR_TIME                 86292
WHEELS_ON                86292
TAXI_IN                  86291
TAXI_OUT                 82912
WHEELS_OFF               82912
DEP_TIME                 80102
DEP_DELAY                80102
FL_DATE                      0
OP_CARRIER                   0
ORIGIN                       0
OP_CARRIER_FL_NUM            0
dtype: int64

In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5918486 entries, 0 to 6429337
Data columns (total 34 columns):
 #   Column               Dtype         
---  ------               -----         
 0   FL_DATE              datetime64[ns]
 1   OP_CARRIER           object        
 2   OP_CARRIER_FL_NUM    int32         
 3   ORIGIN               object        
 4   DEST                 object        
 5   CRS_DEP_TIME         int32         
 6   DEP_TIME             float32       
 7   DEP_DELAY            float32       
 8   TAXI_OUT             float32       
 9   WHEELS_OFF           float32       
 10  WHEELS_ON            float32       
 11  TAXI_IN              float32       
 12  CRS_ARR_TIME         int32         
 13  ARR_TIME             float32       
 14  ARR_DELAY            float32       
 15  CANCELLED            float32       
 16  CANCELLATION_CODE    object        
 17  DIVERTED             float32       
 18  CRS_ELAPSED_TIME     float32       
 19  ACTUAL_ELAPSED_TIME  float

In [29]:
#Save Cleaned Dataset
!pip install pyarrow



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
df.to_parquet("D:\Aircraft-Maintenance-Analytics\data\processed\2009_cleaned.parquet",index=False)

In [32]:
#Verify Save
clean_df = pd.read_parquet("D:\Aircraft-Maintenance-Analytics\data\processed\2009_cleaned.parquet")

clean_df.shape


(5918486, 34)